In [2]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from torchsummary import summary
from torchviz import make_dot
from tqdm import tqdm
import csv
from scipy.sparse import issparse
from sklearn.metrics import roc_auc_score


from model.dataset import BagsDataset, custom_collate_fn
from  model.model import MIL, EarlyStopping
from model.dataset import BagsDataset, custom_collate_fn

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    gpu_name = torch.cuda.get_device_name(torch.cuda.current_device())
    print(f"Using device: {device} ({gpu_name})")
else:
    print(f"Using device: {device}")

Using device: cuda (Tesla V100-PCIE-32GB)


In [4]:
all_genes = pd.read_csv('human.csv')
all_genes = all_genes['Gene'].values
dataset = BagsDataset('training.csv', immune_cell='tcell')
train_size = int(0.7 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
#dataset = load_datasets(data_dir='/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE', immune_cell='tcell',radius=200,)

Processing: adata=HumanLungCancerFFPE.h5ad, radius=200, resolution=low
Processing: adata=UKF275.h5ad, radius=200, resolution=high
Processing: adata=TumE2.h5ad, radius=200, resolution=high
Processing: adata=TumA1.h5ad, radius=200, resolution=high
Processing: adata=p16.h5ad, radius=200, resolution=high
Processing: adata=T_cell.h5ad, radius=30, resolution=high


Creating Bags with radius 30: 100%|███████████████████████| 137051/137051 [00:39<00:00, 3427.96it/s]

Total bags created: 79502
Average instances per bag: 9


In [5]:
dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)



In [6]:
model = MIL(all_genes).to(device)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


In [7]:
num_epochs = 1
patience = 5
early_stopping = EarlyStopping(patience=patience, delta=0.0001)

In [8]:
ig_scores_before_training = torch.sigmoid(model.immunogenicity.ig)
with open('ig_scores_before_training.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Gene', 'IG Score Before Training'])
    for gene, score in zip(all_genes, ig_scores_before_training):
        writer.writerow([gene, score.item()])

In [9]:
for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    
    with tqdm(dataloader, unit="batch") as tepoch:
        for i, (distances, gene_expressions, label, core_idx, current_genes) in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")

            optimizer.zero_grad()

            distances = torch.stack(distances).to(device)
            gene_expressions = torch.stack(gene_expressions).to(device)
            label = label.clone().detach().float().to(device)
            
            output = model(distances, gene_expressions, list(current_genes[0]))
            
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            tepoch.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')
    

    model.eval()
    val_loss = 0.0
    val_predictions = []
    val_labels = []
    with torch.no_grad():
        for val_distances, val_gene_expressions, val_label, _, val_current_genes in val_dataloader:
            val_distances = torch.stack(val_distances).to(device)
            val_gene_expressions = torch.stack(val_gene_expressions).to(device)
            val_label = val_label.clone().detach().float().to(device)
            val_output = model(val_distances, val_gene_expressions, list(val_current_genes[0]))
            val_loss += criterion(val_output, val_label).item()
            val_predictions.extend(val_output.cpu().numpy())
            val_labels.extend(val_label.cpu().numpy())
    
    val_loss /= len(val_dataloader)
    val_auroc = roc_auc_score(val_labels, val_predictions)
    print(f'Validation Loss: {val_loss:.4f}, Validation AUROC: {val_auroc:.4f}')
    
    early_stopping(val_loss, model, epoch)
    if early_stopping.early_stop:
        print(f'Early stopping at epoch {epoch+1}')
        break


Epoch 1/1: 100%|██████████| 55651/55651 [03:56<00:00, 234.84batch/s, loss=0.492]


Epoch [1/1], Loss: 0.6415
Validation Loss: 0.6243, Validation AUROC: 0.3210


In [1]:
ig_scores_after_training = torch.sigmoid(model.immunogenicity.ig)


NameError: name 'torch' is not defined

In [ ]:
ig_score = {
    'Gene': all_genes,
    'IG Score Before Training': [score.item() for score in ig_scores_before_training],
    'IG Score After Training': [score.item() for score in ig_scores_after_training]
}  
df = pd.DataFrame(ig_score)

In [ ]:
df['Difference'] = df['IG Score After Training'] - df['IG Score Before Training']


In [ ]:
df = df.sort_values(by='IG Score After Training', ascending=False)


In [ ]:
df['rank'] = df['Difference'].rank(ascending=False)


In [ ]:
df.head(10) 

In [ ]:
df.to_csv('./changes.csv', index=False)

torch.save(model.state_dict(), './final_model.pth')

In [ ]:
tumor_antigen = pd.read_csv('tumor_antigen.csv')


In [ ]:
common_genes = pd.merge(df, tumor_antigen[['Gene']], on='Gene')


In [ ]:
common_genes
